# 02 逐图理解：5张主图 + 7张补图

先运行两个初始化单元，再任选一图。图形单元直接展示原论文的真实筛选/绘图代码，输出到figures/notebook。建议每图先预测变化方向，再运行，回答改变的是疾病率、人口份额还是基准定义。

In [ ]:
from pathlib import Path
import sys
here = Path.cwd().resolve()
ROOT = next(p for p in [here, *here.parents] if (p / "docs/20261030_INPUT_CHECKSUMS.csv").exists())
sys.path.insert(0, str(ROOT / "code"))
import reproduce as rep
import numpy as np
import pandas as pd
print("复现包：", ROOT.name)


In [ ]:
from notebook_support import figure_inputs
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import display, Image
SOURCE = "recomputed"  # 可改为reference学画图，但不是原始数据复现
a,d,sens,shifts,ez,ec,dyn = figure_inputs(ROOT,source=SOURCE)
print("图表数据来源:",SOURCE)
PCOS = "Polycystic ovarian syndrome"
TYPES,AGES = ["Total","Primary","Secondary"],list(range(15,50,5))
COLORS = ["#19566c","#ba572a","#6b8990"]
signed = lambda x,digits=2:f"{x:+.{digits}f}"
ss = sens[(sens.cause==PCOS)&(sens.subtype=="Total")&(sens.start==1990)&(sens.end==2023)&(sens.source=="GBD")]
labs = ss.scenario.str.replace("omit_","Without ",regex=False).replace({"main":"Main","ages20_49":"Ages 20-49"})
OUT = ROOT / "figures/notebook"
OUT.mkdir(parents=True,exist_ok=True)
def savefig(fig,name):
    fig.savefig(OUT/(name+".pdf"),bbox_inches="tight")
    fig.savefig(OUT/(name+".png"),dpi=180,bbox_inches="tight")
    display(Image(filename=str(OUT/(name+".png"))))
    plt.close(fig)
plt.rcParams.update({"font.family":"DejaVu Sans","font.size":10,"pdf.fonttype":42,"axes.spines.top":False,"axes.spines.right":False})


## 图1：总体轨迹与率差

先看中国与合并参照的水平，再看差值。中国下降不必然意味着差值下降，参照也在变化。

PNG/PDF: figures/notebook/Figure1_PCOS_trends

In [ ]:
plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 10, 'pdf.fonttype': 42, 'axes.spines.top': False, 'axes.spines.right': False})
p = dyn[(dyn.cause == PCOS) & (dyn.subtype == 'Total')]
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5), layout='constrained')
ax[0].plot(p.year, p.china_aspr, label='China', color=COLORS[0])
ax[0].plot(p.year, p.ref_aspr, label='18-country reference', color=COLORS[1])
ax[0].legend(frameon=False)
ax[0].set(title='A  PCOS-attributed total infertility', ylabel='Standardized rate per 100,000', xlabel='Year')
ax[1].plot(p.year, p.gap, color=COLORS[0])
ax[1].axhline(0, color='0.6', lw=0.7)
ax[1].set(title='B  China minus reference', ylabel='Rate difference per 100,000', xlabel='Year')
savefig(fig, 'Figure1_PCOS_trends')

## 图2：原发与继发亚型

总量可能掩盖亚型不同趋势。两亚型用相同全部育龄女性分母和权重，可与总率加和一起理解。

PNG/PDF: figures/notebook/Figure2_PCOS_subtypes

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5), layout='constrained')
for k, t in enumerate(['Primary', 'Secondary']):
    z = dyn[(dyn.cause == PCOS) & (dyn.subtype == t)]
    ax[k].plot(z.year, z.china_aspr, color=COLORS[0], label='China')
    ax[k].plot(z.year, z.ref_aspr, color=COLORS[1], label='18-country reference')
    ax[k].legend(frameon=False)
    ax[k].set(title=t + ' infertility', xlabel='Year', ylabel='Standardized rate per 100,000')
savefig(fig, 'Figure2_PCOS_subtypes')

## 图3：三项对称分解

每组X+Q+S等于黑色菱形净变化。正负号须按中国率、参照率及参照份额各项定义解释。

PNG/PDF: figures/notebook/Figure3_decomposition

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.4), layout='constrained')
xx = np.arange(3)
for k, (c, lab) in enumerate(zip(['china_rate_effect', 'reference_rate_effect', 'reference_country_composition_effect'], ['China rates (X)', 'Reference rates (Q)', 'Reference shares (S)'])):
    z = d[(d.cause == PCOS) & (d.source == 'GBD') & (d.end == 2023)].set_index('subtype').loc[TYPES]
    bars = ax.bar(xx + (k - 1) * 0.24, z[c], width=0.24, color=COLORS[k], label=lab)
    ax.bar_label(bars, fmt='%+.2f', fontsize=8, padding=3)
ax.scatter(xx, z.net_gap_change, color='#202c33', marker='D', s=35, label='Net change', zorder=5)
ax.axhline(0, color='0.5', lw=0.7)
ax.set(xticks=xx, xticklabels=TYPES, ylabel='Contribution per 100,000', ylim=(-270, 340))
ax.legend(ncol=2, frameon=False, loc='upper right', fontsize=9)
savefig(fig, 'Figure3_decomposition')

## 图4：GBD与UN人口参照

替换人口份额、保留GBD病率。相近结果是人口权重稳健性检查，不能称独立疾病验证。

PNG/PDF: figures/notebook/Figure4_population_source

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.8), layout='constrained')
for k, source in enumerate(['GBD', 'UN-WPP2024']):
    z = a[(a.cause == PCOS) & (a.subtype == 'Total') & (a.source == source) & (a.benchmark == 'dynamic')]
    ax[0].plot(z.year, z.gap, label=source, color=COLORS[k], ls=['-', '--'][k], lw=2)
    v = d[(d.cause == PCOS) & (d.subtype == 'Total') & (d.source == source) & (d.end == 2023)].iloc[0]
    ax[1].bar(np.arange(3) + (k - 0.5) * 0.32, [v.china_rate_effect, v.reference_rate_effect, v.reference_country_composition_effect], width=0.32, label=source, color=COLORS[k])
ax[0].set(xlabel='Year', ylabel='China minus reference, per 100,000', title='A  Population-source comparison')
ax[0].legend(frameon=False)
ax[1].axhline(0, color='0.5', lw=0.7)
ax[1].set(xticks=range(3), xticklabels=['China\nrates', 'Reference\nrates', 'Reference\nshares'], ylabel='Contribution per 100,000', title='B  Symmetric decomposition')
ax[1].legend(frameon=False)
savefig(fig, 'Figure4_population_source')

## 图5：当年/固定/等国基准

相同2023疾病率，更换参照基准就可能改变差值符号。三种基准不是同义替代。

PNG/PDF: figures/notebook/Figure5_benchmark_sensitivity

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.1), layout='constrained')
labels = ['Current-year population shares', 'Fixed-1990 population shares', 'Equal country weights']
for k, source in enumerate(['GBD', 'UN-WPP2024']):
    z = a[(a.cause == PCOS) & (a.subtype == 'Secondary') & (a.source == source) & (a.year == 2023)].set_index('benchmark').loc[['dynamic', 'fixed1990', 'equal']]
    bars = ax.barh(np.arange(3) + (k - 0.5) * 0.3, z.gap, height=0.3, label=source, color=COLORS[k])
    ax.bar_label(bars, labels=[signed(x) for x in z.gap], padding=5, fontsize=9)
ax.axvline(0, color='0.5', lw=0.8)
ax.set(yticks=range(3), yticklabels=labels, xlim=(-155, 40), xlabel='China minus reference, per 100,000')
ax.invert_yaxis()
ax.legend(frameon=False, loc='lower center', bbox_to_anchor=(0.5, 1.01), ncol=2)
savefig(fig, 'Figure5_benchmark_sensitivity')

## 补图S1：删一参照国与年龄范围

删除一国后在年龄内重新归一化其余17国；20–49也重新归一化年龄权重。画的是2023端点差。

PNG/PDF: figures/notebook/FigureS1_reference_exclusions

In [ ]:
ss = sens[(sens.cause == PCOS) & (sens.subtype == 'Total') & (sens.start == 1990) & (sens.end == 2023) & (sens.source == 'GBD')]
fig, ax = plt.subplots(figsize=(8, 6.5), layout='constrained')
labs = ss.scenario.str.replace('omit_', 'Without ', regex=False).replace({'main': 'Main', 'ages20_49': 'Ages 20-49'})
ax.barh(labs, ss.end_gap, color=COLORS[0])
ax.invert_yaxis()
ax.axvline(0, color='0.5', lw=0.7)
ax.set(xlabel='2023 total PCOS difference per 100,000')
savefig(fig, 'FigureS1_reference_exclusions')

## 补图S2：三基准年度轨迹

看34年轨迹，避免只看2023。UN仅用于人口份额，仍是同一GBD模型率。

PNG/PDF: figures/notebook/FigureS2_benchmark_trajectories

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.8), layout='constrained')
for k, t in enumerate(['Primary', 'Secondary']):
    for j, (bench, lab) in enumerate([('dynamic', 'Current-year'), ('fixed1990', 'Fixed 1990'), ('equal', 'Equal countries')]):
        z = a[(a.cause == PCOS) & (a.subtype == t) & (a.source == 'UN-WPP2024') & (a.benchmark == bench)]
        ax[k].plot(z.year, z.gap, label=lab, color=COLORS[j])
    ax[k].axhline(0, color='0.5', lw=0.7)
    ax[k].legend(frameon=False, fontsize=8)
    ax[k].set(title=t + ' infertility', xlabel='Year', ylabel='Difference per 100,000')
savefig(fig, 'FigureS2_benchmark_trajectories')

## 补图S3：构成项与有符号净变化

100S/有符号净变化可不在0–100%。50%虚线不是显著性阈值，39.8%–67.9%为设定范围，不是可信区间。

PNG/PDF: figures/notebook/FigureS3_composition_sensitivity

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6.5), layout='constrained')
ax.barh(labs, ss.composition_share_of_net * 100, color=COLORS[2])
ax.axvline(50, color='0.4', ls='--', lw=0.7)
ax.invert_yaxis()
ax.set(xlabel='100 x S / signed net change (%)', xlim=(0, 80))
savefig(fig, 'FigureS3_composition_sensitivity')

## 补图S4：子宫内膜异位症轨迹

末端接近零不代表历年接近零。三端点标记定位模型末年变化，不能据此推断疫情/临床机制。

PNG/PDF: figures/notebook/FigureS4_endometriosis_trajectory

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.8), layout='constrained')
z = dyn[(dyn.cause == 'Endometriosis') & (dyn.subtype == 'Total')]
ax.plot(z.year, z.gap, color=COLORS[0])
ax.axhline(0, color='0.5', lw=0.7)
ax.scatter(z[z.year.isin([2021, 2022, 2023])].year, z[z.year.isin([2021, 2022, 2023])].gap, color=COLORS[1])
ax.set(xlabel='Year', ylabel='China minus reference per 100,000')
savefig(fig, 'FigureS4_endometriosis_trajectory')

## 补图S5：参照国构成变化

年龄内各国人口份额用固定标准年龄权重平均再作差，不是实际全龄人口占比或疾病率变化。

PNG/PDF: figures/notebook/FigureS5_country_shifts

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6), layout='constrained')
z = shifts[shifts.source == 'UN-WPP2024'].sort_values('share_change')
ax.barh(z.country, z.share_change * 100, color=COLORS[0])
ax.axvline(0, color='0.5', lw=0.7)
ax.set(xlabel='Change in age-weighted reference share, percentage points')
savefig(fig, 'FigureS5_country_shifts')

## 补图S6：19国末年变动审计

同一模型来源的19国末年输出审计，不等于独立流行病学验证，不为末年突变编造机制。

PNG/PDF: figures/notebook/FigureS6_terminal_country

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6), layout='constrained')
z = ez.sort_values('yoy_percent')
ax.barh(z.country, z.yoy_percent, color=COLORS[1])
ax.axvline(0, color='0.5', lw=0.7)
ax.set(xlabel='2022-2023 modeled endometriosis rate change (%)')
savefig(fig, 'FigureS6_terminal_country')

## 补图S7：年龄×亚型末年贡献

每柱是年龄权重×2022–2023率增量。七年龄×两亚型加和为总率增量，不是病因分数。

PNG/PDF: figures/notebook/FigureS7_terminal_age_type

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4), layout='constrained')
for k, t in enumerate(['Primary', 'Secondary']):
    z = ec.xs(t, level='subtype').reindex(AGES)
    ax.bar(np.arange(7) + (k - 0.5) * 0.35, z.weighted_increment, width=0.35, label=t, color=COLORS[k])
ax.set(xticks=range(7), xticklabels=[f'{x}-{x + 4}' for x in AGES], xlabel='Age group', ylabel='Contribution per 100,000')
ax.legend(frameon=False)
savefig(fig, 'FigureS7_terminal_age_type')

## 总体理解
相同中国疾病率，在不同参照构成/定义下有不同差值；恒等式区分率变化和份额变化。敏感性范围和计算误差均不替代缺失的联合后验区间。本次复现不新增统计确定性。